In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

import statsmodels.formula.api as smf
from sklearn.ensemble import RandomForestClassifier
import numpy as np
from sklearn.linear_model import LinearRegression, LogisticRegression

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, matthews_corrcoef
from kneed import KneeLocator

from matplotlib.lines import Line2D
from matplotlib.patches import Patch

In [2]:
phase2_df=pd.read_csv('/home/imokhtatif/.vscode-server/Chlamy_Project_v2-main/Data/2025_5_phase2.csv',low_memory=False)

In [6]:
# Make sure que tous les mutants sont communs à toutes les plaques sur chaque régime
plates = ['31v1', '31v2', '31v3']
y2_cols = [f'y2_{i}' for i in range(1, 45)]
light_regimes = phase2_df['light_regime'].unique()

all_filtered = []

for regime in light_regimes:
    subset = phase2_df[
        (phase2_df['light_regime'] == regime) &
        (phase2_df['plate'].isin(plates))
    ]

    # Séparer WT / mutants
    wt_subset = subset[subset['mutant_ID'] == 'WT']
    mutants_subset = subset[subset['mutant_ID'] != 'WT']

    # WT : well_ID communs
    wt_wells_by_plate = {
        plate: set(wt_subset[wt_subset['plate'] == plate]['well_id'])
        for plate in plates
    }
    common_wt_wells = set.intersection(*wt_wells_by_plate.values())
    wt_filtered = wt_subset[wt_subset['well_id'].isin(common_wt_wells)].copy()

    # Mutants : mutant_ID communs
    mutants_by_plate = {
        plate: set(mutants_subset[mutants_subset['plate'] == plate]['mutant_ID'])
        for plate in plates
    }
    common_mutants = set.intersection(*mutants_by_plate.values())
    mutants_filtered = mutants_subset[mutants_subset['mutant_ID'].isin(common_mutants)].copy()

    # Recolle pour ce régime
    regime_filtered = pd.concat([wt_filtered, mutants_filtered], ignore_index=True)
    all_filtered.append(regime_filtered)

# DataFrame final filtré pour tout
clean_df = pd.concat(all_filtered, ignore_index=True)
phase2_df1 = clean_df.copy()

In [7]:
from scipy import interpolate
from scipy.stats import rankdata

def normalize_quantiles(A, ties=True):
    A = np.asarray(A, dtype=np.float64)
    n_rows, n_cols = A.shape
    if n_cols == 1:
        return A.copy()

    i = np.linspace(0, 1, n_rows)
    S = np.full((n_rows, n_cols), np.nan)
    nobs = np.zeros(n_cols, dtype=int)
    sort_idx = []

    for j in range(n_cols):
        col = A[:, j]
        not_nan = ~np.isnan(col)
        x = col[not_nan]
        nobs[j] = len(x)
        sort_order = np.argsort(x)
        sorted_x = x[sort_order]

        if nobs[j] < n_rows:
            f = interpolate.interp1d(np.linspace(0, 1, nobs[j]), sorted_x,
                                     bounds_error=False, fill_value="extrapolate")
            S[:, j] = f(i)
        else:
            S[:, j] = sorted_x

        sort_idx.append(np.argsort(np.argsort(col[not_nan])))

    m = np.nanmean(S, axis=1)
    A_out = np.full_like(A, np.nan)

    for j in range(n_cols):
        col = A[:, j]
        not_nan = ~np.isnan(col)

        if ties:
            r = rankdata(col[not_nan], method='average')
            quant_pos = (r - 1) / (nobs[j] - 1)
            f = interpolate.interp1d(i, m, bounds_error=False, fill_value="extrapolate")
            A_out[not_nan, j] = f(quant_pos)
        else:
            ranks = sort_idx[j]
            A_out[not_nan, j] = m[ranks.astype(int)]

    return A_out

In [8]:

plates = ['31v1', '31v2', '31v3']
df_30v =phase2_df1[phase2_df1['plate'].isin(plates)]

group_counts = (
    df_30v.groupby(['plate', 'light_regime', 'mutant_ID', 'mutated_genes'])
    .size()
    .reset_index(name='count')
)

# Step 3: For each plate and light_regime, count how many mutants had 1, 2, ... rows
summary = (
    group_counts.groupby(['light_regime','plate', 'count'])
    .size()
    .reset_index(name='n_mutants')
)

# Optional: Sort for easier reading
summary = summary.sort_values(by=['light_regime','plate', 'count'])

# Show result
summary

,light_regime,plate,count,n_mutants
0,10min-10min,31v1,1,349
1,10min-10min,31v1,7,1
2,10min-10min,31v2,1,349
3,10min-10min,31v2,7,1
4,10min-10min,31v3,1,349
5,10min-10min,31v3,7,1
6,1min-1min,31v1,1,336
7,1min-1min,31v1,7,1
8,1min-1min,31v2,1,336
9,1min-1min,31v2,7,1


## plate 31 20h ML

In [9]:
def quantile_normalize_plate_group(
    df,
    light_regime,
    plates,
    y2_cols,
    wt_n_replicates=21
):
    """
    Quantile-normalize time series data across plates for a given light regime.

    Parameters:
        df: pandas DataFrame (source data)
        light_regime: str (e.g., '20h_ML')
        plates: list of str (e.g., ['31v1', '31v2', '31v3'])
        y2_cols: list of y2 column names (e.g., ['y2_1', ..., 'y2_44'])
        wt_n_replicates: int (number of WT replicates to align across plates)

    Returns:
        A DataFrame with normalized y2 values for the selected light regime and plates.
    """
    df_subset = df[(df['light_regime'] == light_regime) & (df['plate'].isin(plates))].copy()
    df_normalized = df_subset.copy()

    # Get all unique mutant_ID + mutated_genes combos (excluding WT)
    mutant_keys = (
        df_subset[df_subset['mutant_ID'] != 'WT']
        [['mutant_ID', 'mutated_genes']]
        .drop_duplicates()
        .sort_values(['mutant_ID', 'mutated_genes'])
    )

    for timepoint in y2_cols:
        position_values = []
        valid_plate_indices = {}

        for plate in plates:
            subset = df_subset[df_subset['plate'] == plate].copy()

            # ---- WT processing ----
            wt_rows = subset[subset['mutant_ID'] == 'WT'].copy()
            wt_rows = wt_rows.sort_values(['mutant_ID', 'mutated_genes'])

            if wt_rows.shape[0] < wt_n_replicates:
                missing = wt_n_replicates - wt_rows.shape[0]
                wt_values = np.concatenate([
                    wt_rows[timepoint].values,
                    [np.nan] * missing
                ])
                wt_indices = wt_rows.index.tolist() + [None] * missing
            else:
                wt_subset = wt_rows.head(wt_n_replicates)
                wt_values = wt_subset[timepoint].values
                wt_indices = wt_subset.index.tolist()

            # ---- Mutant processing ----
            mutant_values = []
            mutant_indices = []

            for _, row in mutant_keys.iterrows():
                m_id = row['mutant_ID']
                m_gene = row['mutated_genes']
                match = subset[
                    (subset['mutant_ID'] == m_id) &
                    (subset['mutated_genes'] == m_gene)
                ]
                if match.shape[0] == 0:
                    mutant_values.append(np.nan)
                    mutant_indices.append(None)
                else:
                    mutant_values.append(match[timepoint].values[0])
                    mutant_indices.append(match.index[0])

            # Combine WT + mutant
            values = np.concatenate([wt_values, mutant_values])
            indices = wt_indices + mutant_indices

            position_values.append(values)
            valid_plate_indices[plate] = indices

        # Check row count consistency
        lengths = [len(v) for v in position_values]
        if len(set(lengths)) != 1 or 0 in lengths:
            print(f" Skipping {timepoint} due to mismatch or empty data: {lengths}")
            continue

        # Quantile normalize
        matrix = np.column_stack(position_values)
        # print(matrix)
        normalized_matrix = normalize_quantiles(matrix, ties=True)


        # Write results back
        for col_idx, plate in enumerate(plates):
            indices = valid_plate_indices[plate]
            for row_idx, idx in enumerate(indices):
                if idx is not None:
                    df_normalized.loc[idx, timepoint] = normalized_matrix[row_idx, col_idx]

    return df_normalized


In [10]:
plates_31 = ['31v1', '31v2', '31v3']
y2_columns = [f'y2_{i}' for i in range(1, 45)]

phase2_31_20h_ML_normalized = quantile_normalize_plate_group(
    df=phase2_df1,
    light_regime='20h_ML',
    plates=plates_31,
    y2_cols=y2_columns,
    wt_n_replicates=21
)

phase2_31_20h_ML_normalized[['plate', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_columns]


,plate,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,y2_6,...,y2_35,y2_36,y2_37,y2_38,y2_39,y2_40,y2_41,y2_42,y2_43,y2_44
1029,31v1,WT,WT,N03,0.454972,0.480421,0.482671,0.497210,0.479698,0.489492,...,0.434533,0.426585,0.453392,0.459651,0.423684,0.450714,0.428372,0.438067,0.445103,0.444301
1030,31v1,WT,WT,N22,0.447084,0.458017,0.442475,0.425351,0.460219,0.466748,...,0.427683,0.417938,0.436884,0.437413,0.436282,0.438926,0.435304,0.414614,0.421246,0.422593
1031,31v1,WT,WT,N12,0.422518,0.448118,0.457935,0.441784,0.441738,0.473393,...,0.421469,0.413104,0.408883,0.408785,0.426143,0.420531,0.419200,0.447195,0.438095,0.405743
1032,31v1,WT,WT,C22,0.432450,0.473656,0.464653,0.467882,0.487028,0.461550,...,0.397520,0.413623,0.439371,0.422075,0.412222,0.405717,0.436402,0.404225,0.419613,0.414493
1033,31v1,WT,WT,C12,0.456108,0.470345,0.495395,0.505474,0.485525,0.468707,...,0.405862,0.427520,0.427692,0.472902,0.433098,0.421922,0.447293,0.434046,0.426663,0.452394
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2035,31v3,LMJ.RY0402.136684,Cre10.g453900,F06,0.331504,0.368595,0.383992,0.355091,0.383592,0.383593,...,0.296413,0.338643,0.338478,0.294792,0.321137,0.304649,0.322270,0.299903,0.333451,0.309252
2036,31v3,LMJ.RY0402.137801,Cre07.g332901,F08,0.366562,0.416061,0.393536,0.394566,0.429545,0.436878,...,0.345501,0.370015,0.341497,0.338024,0.348681,0.347128,0.380289,0.337642,0.354710,0.375077
2037,31v3,LMJ.RY0402.189319,Cre16.g648750,F09,0.448002,0.480421,0.482671,0.492685,0.476929,0.484141,...,0.471204,0.477781,0.465147,0.466079,0.479306,0.476917,0.473018,0.471333,0.465948,0.475133
2038,31v3,LMJ.RY0402.088269,Cre10.g443150,F10,0.558306,0.577343,0.582745,0.580289,0.542973,0.572736,...,0.517026,0.547848,0.514792,0.526806,0.520746,0.548416,0.533919,0.545011,0.552840,0.523276


In [11]:
plates = ['31v1', '31v2', '31v3']
phase2_31_20h_ML= phase2_df1[(phase2_df1['light_regime'] == '20h_ML') & (phase2_df1['plate'].isin(plates))].copy()
phase2_31_20h_ML[['plate', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_columns]

,plate,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,y2_6,...,y2_35,y2_36,y2_37,y2_38,y2_39,y2_40,y2_41,y2_42,y2_43,y2_44
1029,31v1,WT,WT,N03,0.443255,0.467362,0.465050,0.492951,0.473287,0.483160,...,0.434427,0.432389,0.454387,0.465790,0.428419,0.463882,0.431791,0.445963,0.447290,0.445501
1030,31v1,WT,WT,N22,0.436289,0.444264,0.428671,0.414326,0.450972,0.458228,...,0.427945,0.423274,0.437582,0.446948,0.442437,0.450752,0.437633,0.422492,0.422674,0.425336
1031,31v1,WT,WT,N12,0.411491,0.437968,0.445971,0.429804,0.435919,0.463115,...,0.422254,0.419800,0.409316,0.416138,0.429309,0.430530,0.421243,0.450110,0.440512,0.410524
1032,31v1,WT,WT,C22,0.421249,0.461813,0.452834,0.457356,0.477357,0.452868,...,0.402337,0.419829,0.440100,0.429457,0.419811,0.418373,0.437698,0.414485,0.421063,0.417880
1033,31v1,WT,WT,C12,0.444896,0.458862,0.478136,0.500985,0.476712,0.459359,...,0.410018,0.432852,0.427369,0.473500,0.437320,0.433001,0.447514,0.441333,0.427723,0.452727
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2035,31v3,LMJ.RY0402.136684,Cre10.g453900,F06,0.331072,0.364561,0.381859,0.343945,0.374073,0.376431,...,0.290632,0.327866,0.329679,0.282793,0.308130,0.289122,0.307261,0.286105,0.322120,0.296953
2036,31v3,LMJ.RY0402.137801,Cre07.g332901,F08,0.361725,0.412890,0.390740,0.384875,0.418783,0.431382,...,0.332337,0.359711,0.332150,0.322564,0.335507,0.332426,0.365120,0.323574,0.339684,0.358540
2037,31v3,LMJ.RY0402.189319,Cre16.g648750,F09,0.445805,0.482111,0.479590,0.477836,0.465787,0.481774,...,0.464357,0.480414,0.462939,0.453294,0.472591,0.474130,0.460872,0.467824,0.458687,0.461320
2038,31v3,LMJ.RY0402.088269,Cre10.g443150,F10,0.561686,0.556785,0.578389,0.573057,0.536048,0.575174,...,0.508752,0.538606,0.527654,0.528231,0.535586,0.543449,0.522771,0.537665,0.547099,0.523716


## 31 plate 2h-2h

In [12]:
plates_31 = ['31v1','31v3']
y2_columns = [f'y2_{i}' for i in range(1, 49)]

phase2_31_2h_2h_normalized = quantile_normalize_plate_group(
    df=phase2_df1,
    light_regime='2h-2h',
    plates=plates_31,
    y2_cols=y2_columns,
    wt_n_replicates=21
)

phase2_31_2h_2h_normalized[['plate', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_columns]

,plate,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,y2_6,...,y2_39,y2_40,y2_41,y2_42,y2_43,y2_44,y2_45,y2_46,y2_47,y2_48
2040,31v1,WT,WT,H12,0.234456,0.231776,0.251156,0.255380,0.253267,0.256213,...,0.642059,0.645474,0.180361,0.204387,0.176052,0.205906,0.639705,0.642942,0.645177,0.648026
2041,31v1,WT,WT,C03,0.234726,0.294409,0.273128,0.272214,0.270314,0.294156,...,0.669791,0.662965,0.215132,0.229576,0.257449,0.236038,0.652341,0.685596,0.644050,0.646710
2042,31v1,WT,WT,C12,0.212421,0.200610,0.255153,0.247460,0.259474,0.217252,...,0.622347,0.635446,0.191090,0.247530,0.181870,0.194086,0.632614,0.632656,0.642479,0.637686
2043,31v1,WT,WT,C22,0.187253,0.184644,0.227725,0.192167,0.229195,0.209745,...,0.621088,0.611524,0.187022,0.167378,0.139814,0.217622,0.619871,0.601236,0.606672,0.623296
2044,31v1,WT,WT,N12,0.213061,0.178610,0.226357,0.203232,0.265792,0.278955,...,0.618347,0.631835,0.148017,0.191516,0.196859,0.222015,0.629766,0.614704,0.619747,0.618427
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3097,31v3,LMJ.RY0402.121674,Cre16.g685250,F07,0.160078,0.228571,0.184696,0.099783,0.191259,0.200794,...,0.552523,0.546863,0.079208,0.100589,0.121677,0.146879,0.552796,0.564837,0.547288,0.540398
3098,31v3,LMJ.RY0402.136684,Cre10.g453900,F06,0.192350,0.186592,0.247116,0.204642,0.258957,0.135373,...,0.614622,0.616840,0.182095,0.175067,0.131714,0.173624,0.588370,0.601066,0.601567,0.596204
3099,31v3,LMJ.RY0402.224902,"Cre07.g350250,Cre17.g732533",F05,0.285192,0.267876,0.283480,0.246445,0.266831,0.256907,...,0.640908,0.665914,0.184680,0.162798,0.206224,0.121117,0.640066,0.659170,0.667192,0.649313
3100,31v3,LMJ.RY0402.190758,Cre01.g040950,F04,0.133757,0.149622,0.181413,0.146024,0.183642,0.155623,...,0.612683,0.600023,0.093598,0.048924,0.102109,0.079554,0.626307,0.622868,0.615188,0.609136


In [13]:

plates = ['31v1', '31v2', '31v3']
phase2_31_2h_2h = phase2_df1[
    (phase2_df1['light_regime'] == '2h-2h') &
    (phase2_df1['plate'].isin(plates))
].copy()
phase2_31_2h_2h[['plate', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_columns]

,plate,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,y2_6,...,y2_39,y2_40,y2_41,y2_42,y2_43,y2_44,y2_45,y2_46,y2_47,y2_48
2040,31v1,WT,WT,H12,0.241683,0.241579,0.271405,0.263851,0.267266,0.263246,...,0.647775,0.654133,0.191005,0.215268,0.180137,0.206086,0.645553,0.652138,0.653309,0.657445
2041,31v1,WT,WT,C03,0.242059,0.305462,0.295442,0.284484,0.282555,0.302213,...,0.675063,0.670071,0.223960,0.240187,0.255147,0.234526,0.655660,0.692261,0.652973,0.656005
2042,31v1,WT,WT,C12,0.219158,0.210508,0.276742,0.254586,0.273908,0.222539,...,0.629696,0.642158,0.201868,0.256472,0.185237,0.193065,0.639402,0.642403,0.651010,0.649787
2043,31v1,WT,WT,C22,0.193646,0.195311,0.244847,0.198888,0.240257,0.213949,...,0.628641,0.622366,0.198313,0.177073,0.142151,0.216308,0.626114,0.612095,0.618173,0.634470
2044,31v1,WT,WT,N12,0.219722,0.188251,0.243406,0.210762,0.278537,0.288385,...,0.625502,0.640686,0.157671,0.202167,0.202132,0.221729,0.637111,0.624367,0.630405,0.628990
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3097,31v3,LMJ.RY0402.121674,Cre16.g685250,F07,0.155614,0.218901,0.164119,0.109014,0.181625,0.197300,...,0.540253,0.532974,0.069213,0.082419,0.116002,0.146827,0.542235,0.547897,0.527063,0.523080
3098,31v3,LMJ.RY0402.136684,Cre10.g453900,F06,0.185901,0.176435,0.227782,0.196057,0.245093,0.127115,...,0.607307,0.606869,0.171949,0.163749,0.127729,0.173048,0.580794,0.590241,0.590035,0.583692
3099,31v3,LMJ.RY0402.224902,"Cre07.g350250,Cre17.g732533",F05,0.280936,0.260531,0.264999,0.239497,0.254032,0.249455,...,0.634752,0.658911,0.174738,0.153040,0.202571,0.118615,0.634548,0.650645,0.658011,0.639673
3100,31v3,LMJ.RY0402.190758,Cre01.g040950,F04,0.127541,0.136550,0.161840,0.140941,0.175974,0.149515,...,0.605146,0.587858,0.083126,0.045532,0.096527,0.088422,0.619507,0.613636,0.603363,0.597310


### 31 plate 10min-10min

In [14]:
plates_31 = ['31v2','31v3']
y2_columns = [f'y2_{i}' for i in range(1, 85)]

phase2_31_10min_10min_normalized = quantile_normalize_plate_group(
    df=phase2_df1,
    light_regime='10min-10min',
    plates=plates_31,
    y2_cols=y2_columns,
    wt_n_replicates=21
)

phase2_31_10min_10min_normalized[['plate', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_columns]

,plate,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,y2_6,...,y2_75,y2_76,y2_77,y2_78,y2_79,y2_80,y2_81,y2_82,y2_83,y2_84
5239,31v2,WT,WT,H12,0.233286,0.614004,0.219356,0.619458,0.229472,0.611818,...,0.182308,0.633393,0.612384,0.175240,0.202768,0.617144,0.639405,0.165521,0.190486,0.635560
5240,31v2,WT,WT,C03,0.213664,0.640973,0.237780,0.645055,0.262517,0.646830,...,0.188852,0.651690,0.655435,0.230448,0.234053,0.656780,0.639011,0.187767,0.235771,0.653520
5241,31v2,WT,WT,C12,0.249320,0.637107,0.246494,0.625030,0.248890,0.635935,...,0.213781,0.630679,0.648578,0.178902,0.206673,0.646616,0.642252,0.197193,0.201856,0.647362
5242,31v2,WT,WT,C22,0.196796,0.603959,0.184744,0.627573,0.227748,0.605659,...,0.164235,0.596986,0.610134,0.163047,0.183738,0.596742,0.613642,0.177311,0.207552,0.600911
5243,31v2,WT,WT,N12,0.239299,0.601170,0.263560,0.613662,0.245933,0.601153,...,0.193576,0.622811,0.622721,0.217514,0.210627,0.613785,0.624087,0.214233,0.177344,0.611075
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6295,31v3,LMJ.RY0402.190758,Cre01.g040950,F04,0.120578,0.567318,0.100497,0.569669,0.131368,0.584985,...,0.101990,0.564795,0.550514,0.104266,0.148298,0.574002,0.577840,0.121438,0.132175,0.563382
6296,31v3,LMJ.RY0402.223305,Cre16.g677200,F12,0.238502,0.645118,0.248035,0.663462,0.258542,0.652220,...,0.213159,0.630679,0.652417,0.220451,0.207281,0.661949,0.654397,0.224066,0.215026,0.650496
6297,31v3,LMJ.RY0402.211740,Cre16.g685250,I01,0.188658,0.546059,0.130739,0.555437,0.132770,0.562245,...,0.111943,0.576792,0.536542,0.094107,0.145950,0.545015,0.549699,0.126461,0.130524,0.548874
6298,31v3,LMJ.RY0402.051628,"Cre13.g589400,Cre13.g589350",A02,0.110914,0.551950,0.174880,0.536114,0.110791,0.547591,...,0.175454,0.545783,0.548463,0.168789,0.148081,0.535764,0.547568,0.135288,0.109731,0.557931


In [15]:
plates = ['31v1', '31v2', '31v3']
phase2_31_10min_10min = phase2_df1[
    (phase2_df1['light_regime'] == '10min-10min') &
    (phase2_df1['plate'].isin(plates))
].copy()
phase2_31_10min_10min[['plate', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_columns ]

,plate,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,y2_6,...,y2_75,y2_76,y2_77,y2_78,y2_79,y2_80,y2_81,y2_82,y2_83,y2_84
5232,31v1,WT,WT,N22,0.279201,0.624934,0.239582,0.598317,0.224547,0.589829,...,0.224575,0.623946,0.191244,0.601971,0.231364,0.618155,0.195406,0.621537,0.186276,0.604586
5233,31v1,WT,WT,N12,0.282120,0.648471,0.246585,0.628075,0.245529,0.597647,...,0.203734,0.603417,0.231117,0.616440,0.187946,0.636107,0.198469,0.626972,0.213853,0.620909
5234,31v1,WT,WT,N03,0.317062,0.641617,0.266887,0.648415,0.225980,0.618245,...,0.213137,0.616751,0.201207,0.623531,0.249125,0.612130,0.226584,0.627962,0.196105,0.633640
5235,31v1,WT,WT,C22,0.264309,0.639547,0.237876,0.632314,0.228451,0.629959,...,0.203334,0.615966,0.192343,0.620745,0.201856,0.621038,0.206667,0.635618,0.219014,0.631288
5236,31v1,WT,WT,C03,0.273159,0.643931,0.250675,0.657232,0.239786,0.626736,...,0.230515,0.641569,0.184372,0.646117,0.184237,0.638736,0.206079,0.641864,0.216034,0.645094
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6295,31v3,LMJ.RY0402.190758,Cre01.g040950,F04,0.112368,0.559836,0.088761,0.566098,0.136407,0.579313,...,0.101775,0.564644,0.547046,0.107446,0.155647,0.573692,0.579839,0.133777,0.141650,0.561547
6296,31v3,LMJ.RY0402.223305,Cre16.g677200,F12,0.237157,0.641902,0.249417,0.659804,0.268901,0.650606,...,0.209840,0.631394,0.650333,0.226766,0.216487,0.660796,0.654963,0.238214,0.225608,0.651958
6297,31v3,LMJ.RY0402.211740,Cre16.g685250,I01,0.185970,0.539356,0.129017,0.551160,0.138060,0.555800,...,0.113781,0.576523,0.533213,0.093847,0.153665,0.544892,0.551489,0.138086,0.139290,0.547614
6298,31v3,LMJ.RY0402.051628,"Cre13.g589400,Cre13.g589350",A02,0.104398,0.546321,0.177219,0.530902,0.115949,0.541055,...,0.175546,0.544950,0.546405,0.171568,0.155526,0.533819,0.549485,0.147029,0.118359,0.556407


### 31 plate 1min-1min

In [16]:
plates_31 = ['31v1','31v2']
y2_columns = [f'y2_{i}' for i in range(1, 89)]

phase2_31_1min_1min_normalized = quantile_normalize_plate_group(
    df=phase2_df1,
    light_regime='1min-1min',
    plates=plates_31,
    y2_cols=y2_columns,
    wt_n_replicates=21
)

phase2_31_1min_1min_normalized[['plate', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_columns]

,plate,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,y2_6,...,y2_79,y2_80,y2_81,y2_82,y2_83,y2_84,y2_85,y2_86,y2_87,y2_88
0,31v1,WT,WT,H12,0.235534,0.573018,0.235892,0.550635,0.246034,0.556265,...,0.210988,0.559463,0.234915,0.556369,0.210081,0.571294,0.236618,0.546427,0.214282,0.567276
1,31v1,WT,WT,C03,0.258106,0.583478,0.260628,0.560122,0.277600,0.567929,...,0.254528,0.586596,0.262217,0.580227,0.277268,0.568981,0.256700,0.571868,0.264807,0.564696
2,31v1,WT,WT,C12,0.230027,0.585699,0.233359,0.558251,0.243307,0.530934,...,0.220870,0.545322,0.270949,0.567915,0.235377,0.559985,0.207303,0.564304,0.231541,0.564218
3,31v1,WT,WT,C22,0.214194,0.581792,0.234646,0.548888,0.211001,0.528575,...,0.184576,0.563603,0.222667,0.554997,0.199933,0.567150,0.226012,0.567620,0.198746,0.557751
4,31v1,WT,WT,N12,0.245763,0.581450,0.253196,0.554540,0.235203,0.531405,...,0.163449,0.551304,0.201956,0.527660,0.183988,0.526867,0.214090,0.544305,0.190409,0.541836
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
688,31v2,LMJ.RY0402.116828,Cre06.g295600,F10,0.181919,0.532436,0.186504,0.516220,0.183657,0.519309,...,0.136386,0.502462,0.146969,0.493254,0.166521,0.478273,0.133183,0.472568,0.153226,0.472979
689,31v2,LMJ.RY0402.190655,Cre16.g664400,F09,0.205860,0.555753,0.194823,0.547037,0.165823,0.544168,...,0.170368,0.501356,0.198100,0.524902,0.203629,0.507523,0.177287,0.521055,0.195548,0.512545
690,31v2,LMJ.RY0402.054432,"Cre08.g372100 & Cre08.g372150,Cre13.g590600",F08,0.145077,0.469440,0.180840,0.504780,0.156591,0.476166,...,0.096037,0.484604,0.142907,0.467871,0.149966,0.468190,0.148686,0.470273,0.148935,0.465834
691,31v2,LMJ.RY0402.190758,Cre01.g040950,F07,0.190024,0.524621,0.136874,0.509129,0.162903,0.511868,...,0.144212,0.458751,0.148906,0.489729,0.156267,0.471511,0.120265,0.498268,0.127245,0.481931


In [17]:
plates = ['31v1', '31v2', '31v3']
phase2_31_1min_1min = phase2_df1[
    (phase2_df1['light_regime'] == '1min-1min') &
    (phase2_df1['plate'].isin(plates))
].copy()
phase2_31_1min_1min[['plate', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_columns]

,plate,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,y2_6,...,y2_79,y2_80,y2_81,y2_82,y2_83,y2_84,y2_85,y2_86,y2_87,y2_88
0,31v1,WT,WT,H12,0.228789,0.570741,0.239472,0.546992,0.238869,0.554967,...,0.215941,0.559334,0.235039,0.555619,0.208223,0.564714,0.242797,0.548719,0.217062,0.566172
1,31v1,WT,WT,C03,0.252614,0.580299,0.264333,0.555863,0.270777,0.566764,...,0.258686,0.585063,0.263101,0.582792,0.274291,0.561932,0.258217,0.574456,0.265018,0.563748
2,31v1,WT,WT,C12,0.223361,0.582509,0.235927,0.554451,0.236199,0.530941,...,0.227098,0.545303,0.270754,0.564197,0.234747,0.551376,0.207533,0.565455,0.233114,0.562821
3,31v1,WT,WT,C22,0.207046,0.579010,0.237506,0.545565,0.205780,0.528227,...,0.191332,0.563408,0.222095,0.554087,0.199323,0.559729,0.228634,0.569831,0.200262,0.557732
4,31v1,WT,WT,N12,0.238838,0.578420,0.255603,0.551873,0.228313,0.531356,...,0.172521,0.552503,0.200606,0.527116,0.184030,0.520996,0.215307,0.547161,0.191461,0.539126
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1024,31v3,LMJ.RY0402.213180,Cre08.g384320,K18,0.158199,0.570118,0.206363,0.530986,0.220004,0.544160,...,0.149482,0.507324,0.089024,0.482665,0.131789,0.507174,0.138287,0.500128,0.158460,0.478442
1025,31v3,LMJ.RY0402.189822,Cre12.g503500,K19,0.273220,0.608575,0.262985,0.590934,0.227995,0.575871,...,0.200744,0.563249,0.173896,0.513491,0.220188,0.563184,0.247279,0.563314,0.224436,0.526452
1026,31v3,LMJ.RY0402.147218,Cre12.g516450,K20,0.214351,0.528897,0.199048,0.561290,0.178381,0.540789,...,0.130982,0.483898,0.089804,0.434382,0.139149,0.457621,0.113622,0.476248,0.168069,0.477934
1027,31v3,LMJ.RY0402.046388,Cre11.g476050,K21,0.192260,0.527284,0.100843,0.526692,0.134855,0.571192,...,0.121105,0.503204,0.145712,0.485376,0.173535,0.488222,0.169236,0.482324,0.147241,0.506999


### 31 plate 30s-30s

In [18]:
plates_31 = ['31v1','31v2','31v3']
y2_columns = [f'y2_{i}' for i in range(1, 89)]

phase2_31_30s_30s_normalized = quantile_normalize_plate_group(
    df=phase2_df1,
    light_regime='30s-30s',
    plates=plates_31,
    y2_cols=y2_columns,
    wt_n_replicates=21
)

phase2_31_30s_30s_normalized[['plate', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_columns]

,plate,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,y2_6,...,y2_79,y2_80,y2_81,y2_82,y2_83,y2_84,y2_85,y2_86,y2_87,y2_88
3102,31v1,WT,WT,N03,0.293563,0.598059,0.281836,0.575487,0.266189,0.540314,...,0.284527,0.556385,0.262706,0.543213,0.267410,0.553446,0.266976,0.540481,0.273336,0.553042
3103,31v1,WT,WT,N22,0.223012,0.573255,0.233564,0.545553,0.263484,0.523463,...,0.236143,0.558458,0.239453,0.536803,0.246522,0.565911,0.277705,0.534769,0.247153,0.536279
3104,31v1,WT,WT,N12,0.255447,0.566563,0.249235,0.540852,0.262470,0.525345,...,0.246637,0.539216,0.249071,0.542199,0.236585,0.536631,0.228806,0.547478,0.269325,0.529418
3105,31v1,WT,WT,C22,0.237216,0.554778,0.257473,0.535176,0.252424,0.541286,...,0.235577,0.537939,0.228530,0.551064,0.238732,0.533276,0.231452,0.534206,0.238999,0.531921
3106,31v1,WT,WT,C12,0.281528,0.585696,0.264098,0.558233,0.280375,0.560700,...,0.257290,0.551351,0.274335,0.550282,0.242280,0.549067,0.295621,0.567922,0.233954,0.563693
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4159,31v3,LMJ.RY0402.251798,Cre12.g524750,K22,0.126267,0.492376,0.140269,0.486111,0.171377,0.439661,...,0.140775,0.430317,0.161979,0.420708,0.132886,0.424831,0.141115,0.432480,0.147792,0.411171
4160,31v3,LMJ.RY0402.103172,Cre12.g538200,K23,0.260728,0.601629,0.266437,0.523847,0.276333,0.542698,...,0.249295,0.491839,0.212793,0.518355,0.204781,0.484690,0.225338,0.483569,0.203625,0.516702
4161,31v3,LMJ.RY0402.147482,Cre12.g505100,K24,0.240302,0.551902,0.178437,0.526273,0.212383,0.499060,...,0.230206,0.497469,0.251344,0.463214,0.139310,0.513789,0.202404,0.491792,0.137038,0.464719
4162,31v3,LMJ.RY0402.099716,Cre12.g560700,I02,0.350271,0.618886,0.317602,0.584078,0.323967,0.585329,...,0.267659,0.587710,0.279648,0.578380,0.275470,0.546299,0.286599,0.578858,0.233954,0.572184


In [19]:
plates = ['31v1', '31v2', '31v3']
phase2_31_30s_30s = phase2_df1[
    (phase2_df1['light_regime'] == '30s-30s') &
    (phase2_df1['plate'].isin(plates))
].copy()
phase2_31_30s_30s[['plate', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_columns]

,plate,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,y2_6,...,y2_79,y2_80,y2_81,y2_82,y2_83,y2_84,y2_85,y2_86,y2_87,y2_88
3102,31v1,WT,WT,N03,0.303334,0.605998,0.298784,0.585777,0.285928,0.548803,...,0.306177,0.574578,0.278228,0.554609,0.281616,0.565422,0.283005,0.545118,0.282306,0.558109
3103,31v1,WT,WT,N22,0.235409,0.578199,0.251880,0.554705,0.280288,0.531175,...,0.262742,0.575509,0.254421,0.549930,0.262418,0.576322,0.292125,0.540343,0.260581,0.541357
3104,31v1,WT,WT,N12,0.267277,0.572479,0.268236,0.549777,0.280030,0.533336,...,0.271073,0.556864,0.264639,0.553097,0.251993,0.547311,0.240206,0.551403,0.280806,0.535840
3105,31v1,WT,WT,C22,0.249531,0.559371,0.276530,0.543750,0.268601,0.549707,...,0.261532,0.555005,0.244800,0.559863,0.254615,0.545396,0.243259,0.540146,0.249953,0.538305
3106,31v1,WT,WT,C12,0.294707,0.591087,0.280401,0.567068,0.299072,0.568511,...,0.280335,0.569578,0.292921,0.558924,0.257529,0.560587,0.312541,0.575439,0.243295,0.569274
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4159,31v3,LMJ.RY0402.251798,Cre12.g524750,K22,0.113668,0.485360,0.124156,0.477241,0.151107,0.430033,...,0.123604,0.420354,0.143718,0.408647,0.118906,0.421450,0.127575,0.414045,0.130878,0.409999
4160,31v3,LMJ.RY0402.103172,Cre12.g538200,K23,0.244453,0.592682,0.245623,0.513389,0.254392,0.530969,...,0.231836,0.484687,0.196197,0.508612,0.192116,0.477254,0.209850,0.468807,0.187385,0.506269
4161,31v3,LMJ.RY0402.147482,Cre12.g505100,K24,0.224484,0.544793,0.162768,0.515014,0.193962,0.489422,...,0.211349,0.489937,0.233992,0.455456,0.125812,0.502535,0.187040,0.478574,0.121055,0.459434
4162,31v3,LMJ.RY0402.099716,Cre12.g560700,I02,0.332984,0.611033,0.297111,0.572845,0.301908,0.571847,...,0.246878,0.578789,0.258059,0.563557,0.261123,0.535320,0.268368,0.562615,0.217155,0.562054


### 31 plate 5min-5min

In [20]:
plates_31 = ['31v1','31v3']
y2_columns = [f'y2_{i}' for i in range(1, 89)]

phase2_31_5min_5min_normalized = quantile_normalize_plate_group(
    df=phase2_df1,
    light_regime='5min-5min',
    plates=plates_31,
    y2_cols=y2_columns,
    wt_n_replicates=21
)

phase2_31_5min_5min_normalized[['plate', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_columns]

,plate,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,y2_6,...,y2_79,y2_80,y2_81,y2_82,y2_83,y2_84,y2_85,y2_86,y2_87,y2_88
6307,31v3,WT,WT,H12,0.200651,0.618906,0.206179,0.603589,0.203696,0.596389,...,0.174806,0.585276,0.161283,0.590037,0.138145,0.585957,0.142676,0.583033,0.165745,0.583101
6308,31v3,WT,WT,C03,0.216754,0.617282,0.200413,0.608062,0.223686,0.612742,...,0.168531,0.596220,0.168775,0.594983,0.139900,0.594247,0.176314,0.601683,0.177288,0.580497
6309,31v3,WT,WT,C12,0.222504,0.621435,0.217866,0.598387,0.236348,0.618082,...,0.146630,0.606629,0.151763,0.592233,0.167775,0.596650,0.166886,0.599245,0.176853,0.583963
6310,31v3,WT,WT,C22,0.190258,0.600303,0.192190,0.593170,0.202149,0.581461,...,0.157877,0.575516,0.159155,0.585977,0.140673,0.571789,0.164308,0.574580,0.159404,0.582973
6311,31v3,WT,WT,N12,0.201554,0.612261,0.213032,0.601061,0.197470,0.597293,...,0.167380,0.583524,0.156139,0.578679,0.147162,0.573131,0.165280,0.586222,0.153074,0.574304
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7363,31v1,LMJ.RY0402.180104,Cre13.g584901,K21,0.109279,0.606807,0.133066,0.591762,0.136261,0.541229,...,0.129528,0.519110,0.094947,0.551310,0.076872,0.534171,0.090417,0.546154,0.147593,0.550417
7364,31v1,LMJ.RY0402.206914,Cre16.g658900,K22,0.181989,0.573158,0.163765,0.546689,0.148712,0.598288,...,0.086991,0.549397,0.108014,0.521284,0.037222,0.509871,0.037379,0.492555,0.077043,0.516710
7365,31v1,LMJ.RY0402.172774,Cre16.g675350,K23,0.217637,0.589592,0.209774,0.538168,0.199666,0.583870,...,0.145334,0.567218,0.172120,0.561393,0.196411,0.552630,0.164308,0.538944,0.145275,0.549251
7366,31v1,LMJ.RY0402.140678,Cre17.g728250,K14,0.173611,0.567592,0.146588,0.563605,0.146611,0.561014,...,0.115595,0.511789,0.136897,0.521665,0.158729,0.520764,0.151344,0.518287,0.135459,0.521076


In [21]:
plates = ['31v1', '31v2', '31v3']
phase2_31_5min_5min = phase2_df1[
    (phase2_df1['light_regime'] == '5min-5min') &
    (phase2_df1['plate'].isin(plates))
].copy()
phase2_31_5min_5min[['plate', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_columns]

,plate,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,y2_6,...,y2_79,y2_80,y2_81,y2_82,y2_83,y2_84,y2_85,y2_86,y2_87,y2_88
6300,31v2,WT,WT,N03,0.235252,0.619182,0.235669,0.615029,0.214913,0.611223,...,0.209114,0.584725,0.213959,0.602052,0.213131,0.606985,0.227378,0.603239,0.216289,0.603994
6301,31v2,WT,WT,N22,0.243448,0.615057,0.226249,0.609411,0.210875,0.582339,...,0.191594,0.587901,0.175568,0.591122,0.188823,0.594255,0.196808,0.565319,0.210983,0.583304
6302,31v2,WT,WT,N12,0.259147,0.620748,0.207495,0.596220,0.224658,0.593280,...,0.192872,0.572071,0.204146,0.590447,0.198421,0.589907,0.199259,0.598875,0.224079,0.590076
6303,31v2,WT,WT,C22,0.236910,0.626667,0.188564,0.641431,0.225325,0.567512,...,0.212083,0.595700,0.151733,0.606053,0.209117,0.598840,0.161975,0.578043,0.166238,0.600652
6304,31v2,WT,WT,C12,0.240706,0.627439,0.231853,0.624830,0.219164,0.615053,...,0.192988,0.618760,0.205819,0.609055,0.194745,0.600616,0.206211,0.610202,0.213816,0.620852
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7363,31v1,LMJ.RY0402.180104,Cre13.g584901,K21,0.118366,0.617817,0.144683,0.603577,0.145369,0.565054,...,0.109763,0.541417,0.077006,0.564309,0.065006,0.550996,0.077452,0.559269,0.124036,0.563896
7364,31v1,LMJ.RY0402.206914,Cre16.g658900,K22,0.176843,0.594689,0.165884,0.568280,0.154286,0.606035,...,0.073548,0.566588,0.088248,0.539724,0.028858,0.527364,0.018152,0.515114,0.067868,0.534844
7365,31v1,LMJ.RY0402.172774,Cre16.g675350,K23,0.202378,0.606356,0.203201,0.561792,0.197015,0.597040,...,0.122934,0.577801,0.138555,0.570673,0.164485,0.563193,0.136278,0.553831,0.123256,0.563171
7366,31v1,LMJ.RY0402.140678,Cre17.g728250,K14,0.171540,0.590507,0.152469,0.583014,0.153656,0.581975,...,0.097612,0.535816,0.114716,0.539896,0.136653,0.538495,0.128924,0.536732,0.115645,0.537948


### 31 plate 1min-5min

In [22]:
plates_31 = ['31v1','31v2','31v3']
y2_columns = [f'y2_{i}' for i in range(1, 89)]

phase2_31_1min_5min_normalized = quantile_normalize_plate_group(
    df=phase2_df1,
    light_regime='1min-5min',
    plates=plates_31,
    y2_cols=y2_columns,
    wt_n_replicates=21
)

phase2_31_1min_5min_normalized[['plate', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_columns]

,plate,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,y2_6,...,y2_79,y2_80,y2_81,y2_82,y2_83,y2_84,y2_85,y2_86,y2_87,y2_88
4164,31v1,WT,WT,N22,0.258330,0.643769,0.215076,0.595535,0.214650,0.621537,...,0.206731,0.628096,0.221934,0.621100,0.203401,0.626264,0.197106,0.618050,0.244746,0.644555
4165,31v1,WT,WT,N03,0.265654,0.644325,0.233215,0.626905,0.232932,0.635478,...,0.198631,0.622829,0.235141,0.641188,0.201808,0.623232,0.225954,0.622954,0.191442,0.620486
4166,31v1,WT,WT,N12,0.278255,0.626349,0.204232,0.603756,0.245305,0.617067,...,0.182369,0.621990,0.214995,0.629220,0.200039,0.608918,0.176601,0.610804,0.204082,0.638267
4167,31v1,WT,WT,C22,0.280437,0.614777,0.225381,0.602750,0.236395,0.614882,...,0.178691,0.618290,0.164380,0.617520,0.197899,0.615258,0.209734,0.618418,0.179880,0.615695
4168,31v1,WT,WT,C03,0.285642,0.629510,0.229944,0.627173,0.246668,0.618192,...,0.209915,0.631309,0.200936,0.636507,0.207855,0.641203,0.208308,0.642743,0.215452,0.638822
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5227,31v3,LMJ.RY0402.189822,Cre12.g503500,K19,0.295718,0.636462,0.262867,0.615848,0.200225,0.616434,...,0.201085,0.618050,0.182050,0.630379,0.180334,0.627134,0.213099,0.634456,0.199258,0.626750
5228,31v3,LMJ.RY0402.147218,Cre12.g516450,K20,0.243227,0.674735,0.238274,0.626905,0.177055,0.643142,...,0.143125,0.646652,0.183403,0.653468,0.149286,0.602856,0.174596,0.625498,0.206534,0.649451
5229,31v3,LMJ.RY0402.046388,Cre11.g476050,K21,0.204129,0.613803,0.188571,0.599768,0.177915,0.590833,...,0.177403,0.568680,0.161055,0.582508,0.103851,0.584719,0.183065,0.570307,0.172074,0.599311
5230,31v3,LMJ.RY0402.251798,Cre12.g524750,K22,0.169335,0.544790,0.139632,0.537161,0.143805,0.510889,...,0.051255,0.530612,0.066942,0.548832,0.063239,0.536754,0.078013,0.490181,0.069875,0.516615


In [23]:
plates = ['31v1', '31v2', '31v3']
phase2_31_1min_5min = phase2_df1[
    (phase2_df1['light_regime'] == '1min-5min') &
    (phase2_df1['plate'].isin(plates))
].copy()
phase2_31_1min_5min[['plate', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_columns]

,plate,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,y2_6,...,y2_79,y2_80,y2_81,y2_82,y2_83,y2_84,y2_85,y2_86,y2_87,y2_88
4164,31v1,WT,WT,N22,0.259387,0.645919,0.206719,0.596863,0.205010,0.627201,...,0.199152,0.628233,0.219546,0.618706,0.199244,0.625721,0.190128,0.619648,0.233578,0.644680
4165,31v1,WT,WT,N03,0.267228,0.646428,0.222899,0.628381,0.223067,0.641643,...,0.192677,0.621086,0.230245,0.638352,0.197059,0.621801,0.218729,0.624554,0.183513,0.622161
4166,31v1,WT,WT,N12,0.278263,0.629109,0.194162,0.606283,0.235130,0.621550,...,0.172726,0.620702,0.214100,0.627839,0.195280,0.607369,0.172788,0.612282,0.196046,0.640258
4167,31v1,WT,WT,C22,0.280897,0.618389,0.213917,0.605330,0.225247,0.618572,...,0.167956,0.617977,0.161348,0.615334,0.192420,0.613327,0.203152,0.619703,0.172404,0.616083
4168,31v1,WT,WT,C03,0.289040,0.632044,0.218185,0.628790,0.236632,0.623941,...,0.205417,0.631322,0.201026,0.635052,0.202879,0.638225,0.201035,0.642668,0.207059,0.641136
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5227,31v3,LMJ.RY0402.189822,Cre12.g503500,K19,0.300283,0.638537,0.281648,0.620529,0.220330,0.622252,...,0.213680,0.626487,0.189603,0.635343,0.188605,0.633507,0.230406,0.638417,0.214556,0.631872
5228,31v3,LMJ.RY0402.147218,Cre12.g516450,K20,0.249925,0.674513,0.256105,0.631227,0.197039,0.645334,...,0.154303,0.654512,0.190259,0.662816,0.160967,0.611885,0.189830,0.629778,0.222817,0.653734
5229,31v3,LMJ.RY0402.046388,Cre11.g476050,K21,0.211546,0.615329,0.205898,0.602215,0.197125,0.595204,...,0.191180,0.579673,0.169857,0.589162,0.112059,0.592580,0.200664,0.573952,0.187694,0.603533
5230,31v3,LMJ.RY0402.251798,Cre12.g524750,K22,0.177934,0.545023,0.156889,0.541998,0.156447,0.513837,...,0.051233,0.542663,0.071460,0.554451,0.072472,0.542792,0.090177,0.486226,0.084826,0.523269


In [24]:
phase2_31_quantile1= pd.concat([
    phase2_31_20h_ML_normalized,
    phase2_31_2h_2h_normalized,
    phase2_31_10min_10min_normalized,
    phase2_31_1min_1min_normalized,
    phase2_31_30s_30s_normalized,
    phase2_31_5min_5min_normalized,
    phase2_31_1min_5min_normalized
], ignore_index=True)

In [25]:
phase2_31_quantile1.to_csv('phase2_31_quantile1.csv',index=False)

In [26]:
phase2_31_quantile1.shape

(5959, 467)

In [27]:
plates = ['31v1', '31v2','31v3']
data=phase2_df1[(phase2_df1['plate'].isin(plates))&(phase2_df1['light_regime']!='20h_HL')]
data.shape

(7368, 467)